# 04 · RxRx3-core 论文下游评估

读取 **phenostate-paper-benchmarks** 的结果。三个任务使用共同孔位及论文的 PCA-CS 批次校正。

[原始论文 §4](https://arxiv.org/html/2503.20158v2#S4) · [EFAAR 官方 Notebook](https://github.com/recursionpharma/EFAAR_benchmarking/tree/trunk/RxRx3-core_benchmarks)

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), Path.cwd())
seven_models_dir = root / 'results/downstream_val_all/paper_7_models'
default_dir = seven_models_dir if (seven_models_dir / 'protocol.json').is_file() else root / 'outputs/rxrx3_paper'
RESULT_DIR = Path(os.environ.get('PHENOSTATE_PAPER_RESULTS_DIR', default_dir))
if not (RESULT_DIR / 'protocol.json').is_file():
    raise FileNotFoundError(f'请先运行 phenostate-paper-benchmarks，再将 RESULT_DIR 指向结果目录。当前目录为 {RESULT_DIR}')

protocol = json.loads((RESULT_DIR / 'protocol.json').read_text(encoding='utf-8'))
models = list(protocol['models'])
energy = pd.read_csv(RESULT_DIR / 'energy_summary.csv')
groups = pd.read_csv(RESULT_DIR / 'energy_groups.csv')
dti = pd.read_csv(RESULT_DIR / 'dti_summary.csv', dtype={'dose': str})
relations = pd.read_csv(RESULT_DIR / 'gene_relationships.csv')
display(pd.DataFrame.from_dict(protocol['models'], orient='index').drop(columns='files'))
print(f"共同孔位 {protocol['n_common_wells']:,}/{protocol['n_metadata_wells']:,}，共同对照 {protocol['n_common_controls']:,}")
print(protocol['scope'])

## 1. 扰动信号强度

能量距离衡量重复扰动孔与对照孔的分布差异，按基因或药物汇总，药物合并剂量。误差线是扰动组的 MAD。随机基线使用与模型同维度的高斯向量和相同孔位。右图显示各扰动组能量距离的累积分布。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, max(4, len(models) * 0.55)))
total = energy.loc[energy['kind'].eq('all')].set_index('model').reindex(models)
y = np.arange(len(models))
axes[0].errorbar(total.energy_mean, y, xerr=total.energy_mad, fmt='o', label='Embedding')
axes[0].errorbar(total.random_mean, y, xerr=total.random_mad, fmt='x', label='Random')
axes[0].set(yticks=y, yticklabels=models, xlabel='Mean energy distance ± MAD')
axes[0].invert_yaxis()
axes[0].legend()
for model in models:
    scores = np.sort(groups.loc[groups['model'].eq(model), 'energy'].to_numpy())
    axes[1].step(scores, np.arange(1, len(scores) + 1) / len(scores), where='post', label=model)
axes[1].set(xlabel='Energy distance', ylabel='Fraction of perturbation groups')
axes[1].legend()
plt.tight_layout()
plt.show()
display(energy)

## 2. 药物靶点检索

每种化合物对候选基因打分。曲线是逐药物算指标再平均，阴影为重复负例抽样的标准差。max 是每个药物–基因对跨剂量的最大绝对余弦相似度。虚线表示随机排序基线。

In [ ]:
doses = sorted(set(dti['dose']) - {'max'}, key=float) + ['max']
x = np.arange(len(doses))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, metric in zip(axes, ('ap', 'auroc')):
    for model in models:
        curve = dti.loc[dti['model'].eq(model)].set_index('dose').reindex(doses)
        mean = curve[f'{metric}_mean'].to_numpy()
        sd = curve[f'{metric}_std'].to_numpy()
        line, = ax.plot(x, mean, 'o-', label=model)
        ax.fill_between(x, mean - sd, mean + sd, color=line.get_color(), alpha=0.12)
    random = dti.groupby('dose')[f'random_{metric}_mean'].mean().reindex(doses)
    ax.plot(x, random, '--', color='gray', label='Random')
    ax.set(xticks=x, xticklabels=doses, xlabel='Concentration (µM) / max', ylabel=metric.upper())
    ax.tick_params(axis='x', labelrotation=30)
    ax.legend()
plt.tight_layout()
plt.show()
display(dti.loc[dti['dose'].eq('max'), ['model', 'n_compounds', 'ap_mean', 'ap_std', 'ap_median_mean', 'ap_z', 'auroc_mean', 'auroc_std']])

## 3. 基因功能关系召回

每个数据库单独汇总。颜色表示已知基因对落在全部基因对余弦相似度两端各 5% 的比例。格内同时显示可评估基因对数量。空值表示当前孔位范围内缺少重叠注释。

In [ ]:
sources = relations['source'].drop_duplicates().tolist()
recall = relations.pivot(index='model', columns='source', values='recall').reindex(index=models, columns=sources)
counts = relations.pivot(index='model', columns='source', values='n_evaluable').reindex_like(recall)
fig, ax = plt.subplots(figsize=(8, max(2.8, 0.75 * len(models))))
image = ax.imshow(recall, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax.set(xticks=range(len(sources)), xticklabels=sources, yticks=range(len(models)), yticklabels=models)
for i, j in np.ndindex(recall.shape):
    score = recall.iat[i, j]
    text = 'NA' if pd.isna(score) else f'{score:.3f}\n(n={counts.iat[i, j]:,})'
    ax.text(j, i, text, ha='center', va='center', color='white' if score > 0.55 else 'black')
fig.colorbar(image, ax=ax, label='Relationship recall')
plt.tight_layout()
plt.show()
display(relations)